# Backtest Lie Detector — Primary Report

**Benchmarking LLMs as point-in-time auditors for financial research.**

*Mohit Apte, Anya Zakharov, Lucie Martin, Myra Singh*
*UChicago FINM 33200 — Generative and Agentic AI for Finance, Spring 2026*

---

This notebook is the executable companion to [`writeup/index.md`](../writeup/index.md).
It walks through the entire benchmark pipeline end-to-end — from constructing
ground-truth cases out of CRSP/Compustat/EDGAR documentation, to running model
evaluations under multiple prompting strategies, to scoring and visualizing the
results across 20 (model × strategy) configurations on 168 distinct cases.

**Headline findings.** Across 4 model variants and 5 prompting strategies, the
benchmark surfaces three robust patterns: **GPT-4o + chain-of-thought** achieves
the highest overall accuracy (85.1% on V5); **Claude Sonnet 4.5** catches 100% of
the 16 subtle false-valid traps under every prompting strategy; and overcaution —
flagging valid workflows as broken — is the dominant failure mode for every
configuration, persisting even on V6 cases stripped of finance-domain content.

**Execution.** The notebook loads pre-committed result JSONLs for the headline
tables and embeds the committed figures inline. A live-demo cell in §7 runs
5 representative cases through the real evaluation pipeline; it falls back to
`MockClient` if no `OPENAI_API_KEY` is present, so the notebook is fully
runnable in any environment.

## Setup

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Find repo root and put the package on the path.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "reproduction"))

import json
from collections import defaultdict

import jsonlines
import pandas as pd
from IPython.display import Image, Markdown, display

# Project package
from backtest_lie_detector.schemas import (
    BenchmarkCase, ModelResponse, ScoredResponse, Validity, ViolationType, Module,
)
from backtest_lie_detector.benchmark.build_cases import (
    load_benchmark, generate_v5_cases, get_chronology_cases_only, get_code_cases_only,
)
from backtest_lie_detector.evals.prompts import (
    get_system_prompt, get_violation_descriptions, format_benchmark_prompt,
)
from backtest_lie_detector.evals.model_clients import (
    MockClient, OpenAIClient, parse_model_json,
)
from backtest_lie_detector.evals.run_eval import run_single_case
from backtest_lie_detector.evals.scoring import (
    score_response, aggregate_scores, compute_calibration_metrics,
    score_validity, score_violations, score_severity_weighted, score_repair_heuristic,
    SEVERITY_WEIGHTS,
)
from backtest_lie_detector.schemas import EvaluationConfig
from backtest_lie_detector.analysis.plots import setup_plot_style

# AA-Omniscience OI helper (reuse from scripts/reproduction)
from run_aa_omniscience import aa_omniscience_index

setup_plot_style()
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 50)

# Paths
DATA_DIR = REPO_ROOT / "data" / "benchmark"
RESULTS_DIR = REPO_ROOT / "outputs" / "results"
FIGURES_DIR = REPO_ROOT / "outputs" / "figures"

# Existence sanity check — fail loudly if anything is missing.
required_files = [
    DATA_DIR / "benchmark_v5.jsonl",
    DATA_DIR / "benchmark_v6.jsonl",
    RESULTS_DIR / "prompting_cot.jsonl",
    RESULTS_DIR / "claude_sonnet46_minimal_v5.jsonl",
    RESULTS_DIR / "v6_chronology_claude_sonnet.jsonl",
    RESULTS_DIR / "code_cases_claude_sonnet.jsonl",
    RESULTS_DIR / "aa_omniscience_gpt4o_generic.jsonl",
    FIGURES_DIR / "all_accuracy_by_strategy.png",
    FIGURES_DIR / "all_safety_vs_overcaution.png",
    FIGURES_DIR / "all_strategy_trends.png",
    FIGURES_DIR / "all_difficulty_heatmap.png",
    FIGURES_DIR / "all_v6only_accuracy.png",
]
missing = [str(p) for p in required_files if not p.exists()]
assert not missing, f"Missing required files: {missing}"

print(f"Repo root: {REPO_ROOT}")
print(f"All {len(required_files)} required files present.")

Repo root: /Users/myrasingh/Documents/uchicago/finm_masters/FINM33200/Backtest-Lie-Detector
All 12 required files present.


## §1 Motivation: Point-in-Time Validity Errors

A backtest can run cleanly, produce reasonable-looking returns, and still be
silently invalid because it used information that was not yet public at the
historical decision point. These "point-in-time" (PIT) errors come in
predictable forms: a researcher uses the ticker `META` for a 2018 Facebook
event (META did not exist until 2022 — the ticker was `FB`); a trading
strategy executes at 3:55 PM ET based on an 8-K that EDGAR did not accept
until 4:07 PM ET; a value strategy filters to companies still listed today,
silently excluding every firm that went bankrupt during the backtest period.

These errors are hard to catch by code review because the code itself is
correct — the bug lives in the *temporal validity* of the inputs. As LLMs
are increasingly used to review and audit financial research workflows, we
built a benchmark to measure whether they can actually catch this class of
error. The four modules of the benchmark — *Ticker Time Machine*,
*Filing Clock*, *Accounting Availability*, and *Survivorship & Delisting* —
each target a distinct family of PIT mistakes documented in CRSP, SEC EDGAR,
Compustat, and academic literature.

## §2 The Schema: What Constitutes a Test Case

Every test case is a `BenchmarkCase` (Pydantic model): a natural-language
workflow description, an `expected_validity` label (`valid` / `invalid` /
`ambiguous`), a list of `expected_violations` drawn from a fixed enum of 17
violation types, and a list of `expected_repair` steps that a competent
auditor would propose. The model's task is to return a `ModelResponse` JSON
object with the same fields, which is then scored against the ground truth.

In [2]:
# Show one real case as the canonical example of the schema.
v5_cases = generate_v5_cases()
example_case = next(c for c in v5_cases if c.id == "ttm_meta_2018_cambridge")

# Pretty-print the case as JSON, dropping internal fields for brevity.
case_json = example_case.model_dump(mode="json")
case_json.pop("source_fields", None)
case_json.pop("case_tags", None)
print(json.dumps(case_json, indent=2))

{
  "id": "ttm_meta_2018_cambridge",
  "module": "ticker_time_machine",
  "difficulty": "easy",
  "prompt": "A researcher studies the Cambridge Analytica event on 2018-03-20 and queries daily returns using ticker META. Audit this workflow. Is it point-in-time valid? If not, identify the issue and propose a repair.",
  "expected_validity": "invalid",
  "expected_violations": [
    "identifier_time_travel"
  ],
  "expected_repair": [
    "Use the historical ticker valid on 2018-03-20, not the modern ticker.",
    "The correct ticker was FB on 2018-03-20.",
    "Resolve the security using a point-in-time identifier such as CRSP PERMNO."
  ],
  "ground_truth_notes": "Meta Platforms did not trade under ticker META on 2018-03-20. The company was still Facebook Inc. trading under ticker FB. META became the ticker on June 9, 2022, after the corporate rebrand.",
  "requires_data_validation": false,
  "data_source": "manual",
  "source_note": ""
}


In [3]:
# List the 17 violation types the model can flag, plus severity weights.
# Higher severity weight = greater penalty for missing this violation type.
descriptions = get_violation_descriptions()
rows = []
for v in ViolationType:
    rows.append({
        "violation_type": v.value,
        "severity_weight": SEVERITY_WEIGHTS.get(v, 1),
        "description": descriptions.get(v, "(V5+ implementation-bug type)"),
    })
pd.DataFrame(rows)

,violation_type,severity_weight,description
0,identifier_time_travel,2,"Using a stock ticker, CUSIP, or other identifier that was not valid for the historical period be..."
1,issuer_security_confusion,2,"Confusing issuer-level and security-level identifiers, or failing to track the correct security ..."
2,filing_clock_leakage,3,Using information from an SEC filing before the filing was publicly available on EDGAR.
3,accounting_availability_leakage,3,Using accounting data before the financial statements containing that data were publicly filed.
4,restatement_leakage,3,Using restated financial data for a period before the restatement was filed.
5,survivorship_bias,3,"Including only companies that survived to the present, excluding companies that delisted, merged..."
6,delisting_return_omission,3,"Failing to include delisting returns when a company stops trading, especially for distressed del..."
7,wrong_event_window,2,Using an event window that does not properly align with when the event information became public.
8,timezone_error,2,"Incorrectly handling timezone conversions, especially for filings or trading across different ti..."
9,link_date_leakage,1,(V5+ implementation-bug type)


## §3 Building the Benchmark: V1 → V6

The benchmark evolved over six iterations, each adding cases that targeted
a specific weakness exposed by the prior version. Case generators are pure
Python — running them is deterministic and produces the JSONL files in
`data/benchmark/`. The table below describes the progression.

| Version | Cases | Main addition |
|---|---:|---|
| V1 | 42 | Original hand-curated seeds across 4 modules |
| V2 | 69 | +17 adversarial + 10 WRDS-backed cases |
| V3 | 105 | +36 hard, source-backed cases (13 families) |
| V4 | 125 | +20 calibration cases (trap-valid, ambiguous) |
| V5 | 141 | +16 false-valid traps (subtle implementation bugs) |
| V6 | 156 + 12 code | V5 + 15 chronology cases; code cases evaluated as a separate battery |

V5 is the main benchmark; V6 adds two methodology contributions designed to
isolate temporal reasoning from finance-domain knowledge. The code-auditing
battery is intentionally separate from `benchmark_v6.jsonl` and is evaluated
through `run_code_cases_eval.py`.

In [4]:
# Regenerate the V5 case list deterministically and report the distribution.
v5_cases = generate_v5_cases()
chrono_cases = get_chronology_cases_only()
code_cases = get_code_cases_only()

print(f"V5 cases:         {len(v5_cases)}")
print(f"V6 chronology:    {len(chrono_cases)}")
print(f"V6 code:          {len(code_cases)}")
print(f"V6 total matrix:  {len(v5_cases) + len(chrono_cases) + len(code_cases)} cases\n")

# Show the structural breakdown of V5.
v5_df = pd.DataFrame([
    {"id": c.id, "module": c.module.value, "difficulty": c.difficulty.value,
     "expected_validity": c.expected_validity.value, "n_violations": len(c.expected_violations)}
    for c in v5_cases
])

print("By module:")
print(v5_df.groupby("module").size().to_string())
print("\nBy expected validity:")
print(v5_df.groupby("expected_validity").size().to_string())
print("\nBy difficulty:")
print(v5_df.groupby("difficulty").size().to_string())

V5 cases:         141
V6 chronology:    15
V6 code:          12
V6 total matrix:  168 cases

By module:
module
accounting_availability    24
filing_clock               31
survivorship_delisting     38
ticker_time_machine        48

By expected validity:
expected_validity
ambiguous    14
invalid      86
valid        41

By difficulty:
difficulty
easy      18
hard      89
medium    34


## §4 Ground Truth: Three Validity Labels, Five Ambiguity Categories

Each case is labeled `valid`, `invalid`, or `ambiguous`. A case is labeled
`ambiguous` *only* when the prompt lacks information required to determine
PIT validity — five categories trigger this label (missing filing acceptance
timestamp, unspecified accounting-lag methodology, unspecified share-class
purpose, missing event decision timestamp, unspecified data-source version).

The hand-curated `ground_truth_notes` field on each case documents *why* the
label is correct. For source-backed cases, the citation is in `source_note`.
The three examples below illustrate the three label categories.

In [5]:
# Pull one case of each validity label and display the ground truth rationale.
v5 = generate_v5_cases()
case_map = {c.id: c for c in v5}

exemplars = [
    "valid_historical_ticker_apple",     # valid control
    "ttm_meta_2018_cambridge",            # invalid - identifier_time_travel
    "cal_ambig_filing_no_timestamp",      # ambiguous - missing timestamp
]
rows = []
for cid in exemplars:
    c = case_map.get(cid)
    if c is None:
        continue
    rows.append({
        "id": c.id,
        "module": c.module.value,
        "expected_validity": c.expected_validity.value,
        "expected_violations": [v.value for v in c.expected_violations],
        "prompt": c.prompt,
        "ground_truth_notes": c.ground_truth_notes,
    })
pd.DataFrame(rows).set_index("id").T

id,valid_historical_ticker_apple,ttm_meta_2018_cambridge,cal_ambig_filing_no_timestamp
module,ticker_time_machine,ticker_time_machine,filing_clock
expected_validity,valid,invalid,ambiguous
expected_violations,[],[identifier_time_travel],[]
prompt,"A researcher studies Apple's iPhone announcement on January 9, 2007 and queries daily returns us...",A researcher studies the Cambridge Analytica event on 2018-03-20 and queries daily returns using...,A researcher studies 8-K filings and same-day stock returns. They state that each 8-K was 'filed...
ground_truth_notes,Apple Inc. has traded under ticker AAPL continuously since going public in 1980. Using AAPL for ...,Meta Platforms did not trade under ticker META on 2018-03-20. The company was still Facebook Inc...,An 8-K 'filed on the event date' could have been accepted at 6:00 AM or 9:00 PM. Without timesta...


## §5 Prompting Strategies

The benchmark evaluates models under five prompting strategies:

| Strategy | System prompt | Few-shot |
|---|---|---:|
| `minimal` | One sentence + JSON schema | 0 |
| `default` | A few bullets naming the common issues | 0 |
| `zero_shot` (a.k.a. `finance_auditor` / "specialized") | Long expert-auditor brief with specific examples | 0 |
| `few_shot` | `finance_auditor` + 3 worked examples | 3 |
| `chain_of_thought` | 5-step reasoning rubric before the JSON answer | 0 |

The five strategies vary along two axes: amount of domain knowledge encoded in
the prompt (minimal → finance_auditor) and presence of structured reasoning
(few_shot's examples, CoT's step-by-step rubric).

In [6]:
# Print the first ~700 chars of each system prompt for comparison.
for name in ["minimal", "default", "finance_auditor", "chain_of_thought"]:
    p = get_system_prompt(name)
    label = "zero_shot (= finance_auditor)" if name == "finance_auditor" else name
    print(f"=== {label} ===")
    print(p[:700] + (" ... (truncated)" if len(p) > 700 else ""))
    print()

=== minimal ===
You are a financial research auditor. Analyze the provided workflow and determine if it has any point-in-time validity issues.

You must respond with a JSON object following this exact schema:

{
  "validity": "valid" | "invalid" | "ambiguous",
  "violations": [
    // Array of violation types from this list:
    // "identifier_time_travel", "issuer_security_confusion", "filing_clock_leakage",
    // "accounting_availability_leakage", "restatement_leakage", "survivorship_bias",
    // "delisting_return_omission", "wrong_event_window", "timezone_error"
  ],
  "explanation": "Concise explanation of your analysis.",
  "repair": [
    "Step 1 to fix the workflow.",
    "Step 2 to fix the workflo ... (truncated)

=== default ===
You are a financial research auditor specializing in data integrity for quantitative finance.

Your task is to audit financial research workflows for point-in-time validity issues. These are errors where a backtest or event study uses information tha

## §6 Model Clients and the Eval Pipeline

`OpenAIClient`, `AnthropicClient`, and `MockClient` all implement the same
`BaseModelClient` interface (`.call(system_prompt, user_prompt, temperature,
max_tokens) -> (raw_text, latency_ms)`). All API clients wrap the call in
`tenacity` with 3 retries and exponential backoff. The raw response text is
then fed to `parse_model_json()`, which is permissive about Markdown code
fences, trailing commas, and single quotes — when the model emits valid JSON
in any reasonable form, parsing succeeds.

The cell below verifies the parser against a synthetic response that
intentionally contains a Markdown fence.

In [7]:
# Demonstrate the JSON parser against a synthetic response that mimics a
# real model output (Markdown-fenced JSON with prose around it).
synthetic_response = '''
Sure, here is my analysis:

```json
{
  "validity": "invalid",
  "violations": ["identifier_time_travel"],
  "explanation": "META was not a valid ticker on 2018-03-20.",
  "repair": ["Use ticker FB for events before June 9, 2022."],
  "confidence": 0.95
}
```
'''
parsed = parse_model_json(synthetic_response)
parsed.model_dump()

{'validity': <Validity.INVALID: 'invalid'>,
 'violations': [<ViolationType.IDENTIFIER_TIME_TRAVEL: 'identifier_time_travel'>],
 'explanation': 'META was not a valid ticker on 2018-03-20.',
 'repair': ['Use ticker FB for events before June 9, 2022.'],
 'confidence': 0.95}

## §7 Live Demo: 5 Cases Through the Real Pipeline

To prove the pipeline works end-to-end without forcing a costly full rerun,
this cell evaluates 5 representative cases through both `MockClient` (always
runs) and `OpenAIClient` (runs if `OPENAI_API_KEY` is set, otherwise skipped).
The 5 cases were chosen to cover all four benchmark modules plus one valid
control.

In [8]:
# Pick 5 representative cases: one per module + one valid control.
demo_ids = [
    "ttm_meta_2018_cambridge",              # ticker time machine — invalid
    "filing_clock_after_close_8k",          # filing clock — invalid
    "acct_fy_end_vs_filing_date",           # accounting availability — invalid
    "survivorship_current_universe_2000_2020",  # survivorship — invalid
    "valid_historical_ticker_apple",        # valid control
]
v5 = generate_v5_cases()
case_map = {c.id: c for c in v5}
demo_cases = [case_map[cid] for cid in demo_ids]

# Always run mock; conditionally run live.
mock_client = MockClient()
live_client = None
if os.getenv("OPENAI_API_KEY"):
    try:
        live_client = OpenAIClient(model="gpt-4o")
        print("Live client: OpenAI gpt-4o (will use API credit).")
    except Exception as e:
        print(f"Live client unavailable ({e}); using mock only.")
else:
    print("OPENAI_API_KEY not set; using mock only.")

mock_config = EvaluationConfig(
    model_name="mock-model",
    config_name="mock-demo",
    provider="openai",
    system_prompt_type="finance_auditor",
)
live_config = EvaluationConfig(
    model_name="gpt-4o",
    config_name="live-demo",
    provider="openai",
    system_prompt_type="finance_auditor",
)

OPENAI_API_KEY not set; using mock only.


In [9]:
# Run each case through the available clients and collect scored responses.
rows = []
for case in demo_cases:
    mock_scored = run_single_case(case, mock_client, mock_config)
    live_scored = run_single_case(case, live_client, live_config) if live_client else None

    row = {
        "case_id": case.id,
        "module": case.module.value,
        "expected": case.expected_validity.value,
        "mock_pred": mock_scored.parsed_response.validity.value if mock_scored.parsed_response else "(parse fail)",
        "mock_correct": mock_scored.validity_correct,
        "mock_latency_ms": round(mock_scored.latency_ms, 1),
    }
    if live_scored is not None:
        row["live_pred"] = live_scored.parsed_response.validity.value if live_scored.parsed_response else "(parse fail)"
        row["live_correct"] = live_scored.validity_correct
        row["live_latency_ms"] = round(live_scored.latency_ms, 1)
    rows.append(row)

demo_df = pd.DataFrame(rows)
demo_df

,case_id,module,expected,mock_pred,mock_correct,mock_latency_ms
0,ttm_meta_2018_cambridge,ticker_time_machine,invalid,invalid,True,100.0
1,filing_clock_after_close_8k,filing_clock,invalid,invalid,True,100.0
2,acct_fy_end_vs_filing_date,accounting_availability,invalid,invalid,True,100.0
3,survivorship_current_universe_2000_2020,survivorship_delisting,invalid,invalid,True,100.0
4,valid_historical_ticker_apple,ticker_time_machine,valid,invalid,False,100.0


## §8 Scoring: From Single Responses to Calibration Metrics

For each scored response, we compute:

- **Validity correctness** — does the predicted label match the expected label?
- **Violation precision / recall / F1** — set comparison on the typed violation tags
- **Severity-weighted recall** — same recall, but weighted (3 for serious
  violations like `filing_clock_leakage`, 2 for ones like `wrong_event_window`)
- **Repair score** — keyword-overlap heuristic against the expected repair text
- **Confidence-when-wrong** — overconfidence tracker

Aggregated across many cases, the most operationally important metrics are
**false-invalid rate** (fraction of *valid* cases the model flagged invalid —
the overcaution measure) and **false-valid rate** (fraction of *invalid*
cases the model approved — the dangerous one). The cell below computes
these on the live-demo responses from §7.

In [10]:
# Re-score the demo cases at the metric level to show what the scoring layer produces.
mock_scored_list = [run_single_case(c, mock_client, mock_config) for c in demo_cases]
agg = aggregate_scores(mock_scored_list)
cal = compute_calibration_metrics(mock_scored_list, demo_cases)

print("Aggregate metrics (MockClient on demo cases):")
for k, v in agg.items():
    print(f"  {k:35s} {v:.3f}" if isinstance(v, float) else f"  {k:35s} {v}")

print("\nCalibration metrics (MockClient on demo cases):")
for k, v in cal.items():
    print(f"  {k:35s} {v:.3f}" if isinstance(v, float) else f"  {k:35s} {v}")

Aggregate metrics (MockClient on demo cases):
  total_cases                         5
  parse_success_rate                  1.000
  validity_accuracy                   0.800
  mean_violation_precision            0.400
  mean_violation_recall               0.400
  mean_violation_f1                   0.400
  mean_severity_weighted_recall       0.400
  mean_repair_score                   0.305
  mean_latency_ms                     100.000

Calibration metrics (MockClient on demo cases):
  false_invalid_rate                  1.000
  false_valid_rate                    0.000
  ambiguous_accuracy                  0.000
  ambiguous_invalid_rate              0.000
  valid_trap_accuracy                 0.000
  overcaution_score                   0.500
  uncertainty_score                   0.000
  n_valid_cases                       1
  n_invalid_cases                     4
  n_ambiguous_cases                   0
  n_trap_valid_cases                  0
  n_false_invalids                    1
  n

## §9 Full V5 Results — 4 Models × 5 Prompting Strategies

The headline results: every combination of 4 model variants (GPT-4o, Sonnet
4.5, Sonnet 4.6, Haiku 4.5) and 5 prompting strategies, evaluated on all 141
V5 cases. The cell below auto-discovers the result JSONL files written by
each evaluation script and reconstructs the leaderboard.

In [11]:
# Helper: load a list of ScoredResponse from a jsonl file.
def load_results_jsonl(path: Path) -> list[ScoredResponse]:
    with jsonlines.open(path) as r:
        return [ScoredResponse.model_validate(rec) for rec in r]


# Define the 20 configurations: 4 models × 5 strategies.
STRATEGIES = ["minimal", "default", "zero_shot", "few_shot", "cot"]
MODELS = [
    ("Haiku 4.5",  "claude_haiku45_{strat}_v5.jsonl"),
    ("Sonnet 4.5", "claude_sonnet45_{strat}_v5.jsonl"),
    ("Sonnet 4.6", "claude_sonnet46_{strat}_v5.jsonl"),
    ("GPT-4o",     "prompting_{strat}.jsonl"),
]

v5 = generate_v5_cases()
rows = []
for model_label, file_pattern in MODELS:
    for strat in STRATEGIES:
        path = RESULTS_DIR / file_pattern.format(strat=strat)
        if not path.exists():
            continue
        results = load_results_jsonl(path)
        agg = aggregate_scores(results)
        cal = compute_calibration_metrics(results, v5)
        rows.append({
            "model":              model_label,
            "strategy":           strat,
            "accuracy":           round(agg["validity_accuracy"] * 100, 1),
            "false_invalid_pct":  round(cal["false_invalid_rate"] * 100, 1),
            "false_valid_pct":    round(cal["false_valid_rate"] * 100, 1),
            "viol_f1":            round(agg["mean_violation_f1"] * 100, 1),
            "repair_score":       round(agg["mean_repair_score"] * 100, 1),
            "mean_latency_ms":    round(agg["mean_latency_ms"], 0),
        })

leaderboard = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
print(f"Loaded {len(leaderboard)} configurations.\n")
print(f"Top 5 by accuracy:")
print(leaderboard.head(5).to_string(index=False))
print(f"\nBest false-valid (safest): {leaderboard.loc[leaderboard['false_valid_pct'].idxmin(), ['model', 'strategy', 'false_valid_pct']].to_dict()}")
print(f"Best false-invalid (least overcautious): {leaderboard.loc[leaderboard['false_invalid_pct'].idxmin(), ['model', 'strategy', 'false_invalid_pct']].to_dict()}")
leaderboard

Loaded 20 configurations.

Top 5 by accuracy:
     model strategy  accuracy  false_invalid_pct  false_valid_pct  viol_f1  repair_score  mean_latency_ms
    GPT-4o      cot      85.1               19.5              0.0     64.8          62.8           2432.0
Sonnet 4.5  default      82.3               31.7              1.2     61.7          74.7          10236.0
    GPT-4o few_shot      82.3                9.8              0.0     67.4          63.4           1658.0
    GPT-4o  default      81.6               29.3              5.8     56.7          58.0           1730.0
Sonnet 4.5 few_shot      81.6               34.1              1.2     64.4          72.9           9147.0

Best false-valid (safest): {'model': 'GPT-4o', 'strategy': 'cot', 'false_valid_pct': 0.0}
Best false-invalid (least overcautious): {'model': 'GPT-4o', 'strategy': 'few_shot', 'false_invalid_pct': 9.8}


,model,strategy,accuracy,false_invalid_pct,false_valid_pct,viol_f1,repair_score,mean_latency_ms
0,GPT-4o,cot,85.1,19.5,0.0,64.8,62.8,2432.0
1,Sonnet 4.5,default,82.3,31.7,1.2,61.7,74.7,10236.0
2,GPT-4o,few_shot,82.3,9.8,0.0,67.4,63.4,1658.0
3,GPT-4o,default,81.6,29.3,5.8,56.7,58.0,1730.0
4,Sonnet 4.5,few_shot,81.6,34.1,1.2,64.4,72.9,9147.0
5,Sonnet 4.6,minimal,81.6,12.2,0.0,56.6,76.6,13262.0
6,Sonnet 4.6,default,80.9,14.6,0.0,59.1,76.2,12856.0
7,Sonnet 4.6,zero_shot,80.9,12.2,1.2,56.2,75.0,15301.0
8,GPT-4o,zero_shot,80.9,31.7,0.0,61.0,60.9,2109.0
9,GPT-4o,minimal,80.1,29.3,5.8,52.6,55.7,1663.0


### Headline figures

The two figures below — produced by `scripts/reproduction/generate_all_figures.py`
from the same JSONLs loaded above — visualize the full 4 × 5 sweep.

In [12]:
# Interactive Plotly version (replaces the static PNG previously embedded here).
from generate_plotly_charts import (
    fig_leaderboard, fig_safety_overcaution, fig_trap_cases,
    fig_v6, fig_aa_omniscience,
    build_leaderboard_df, build_trap_df, build_v6_df, build_aa_df,
)

_df = build_leaderboard_df(RESULTS_DIR)
_fig = fig_leaderboard(_df)
_fig

GPT-4o leads under every strategy; chain-of-thought pushes it to 85.1% — the
single highest cell. Claude Sonnet 4.5 and 4.6 cluster around 77-82% with
relatively little spread across strategies. Haiku 4.5 is the weakest variant
and is the one model that is *hurt* by chain-of-thought (drops to 71%).

In [13]:
# Interactive Plotly version (replaces the static PNG previously embedded here).
from generate_plotly_charts import (
    fig_leaderboard, fig_safety_overcaution, fig_trap_cases,
    fig_v6, fig_aa_omniscience,
    build_leaderboard_df, build_trap_df, build_v6_df, build_aa_df,
)

_df = build_leaderboard_df(RESULTS_DIR)
_fig = fig_safety_overcaution(_df)
_fig

X-axis is overcaution (false-invalid rate); Y-axis is danger (false-valid rate).
Almost every configuration sits below the 2% false-valid safety threshold;
only the two leanest GPT-4o configurations (`minimal` and `default`) cross it
because they approve some of the 16 subtle traps. The Sonnet 4.6 cluster
near the bottom-left (10-15% overcaution, 0-1% false-valid) is the cleanest
calibration region on the plot.

## §10 V5 False-Valid Traps: What Makes a Bug Convincing

V5 added 16 cases designed to *fool* the model: workflows with subtle
implementation bugs hidden behind professional methodology language. They
isolate the question of whether a model can see past reassuring prose. The
cell below restricts every configuration's results to just these 16 cases
and reports trap accuracy + false-valid counts.

In [14]:
# Filter to the 16 false-valid-trap cases and compute per-config trap stats.
trap_cases = [c for c in v5 if "false_valid_trap" in (c.case_tags or [])]
trap_ids = {c.id for c in trap_cases}
print(f"Trap cases: {len(trap_ids)}")

rows = []
for model_label, file_pattern in MODELS:
    for strat in STRATEGIES:
        path = RESULTS_DIR / file_pattern.format(strat=strat)
        if not path.exists():
            continue
        results = [r for r in load_results_jsonl(path) if r.case_id in trap_ids]
        if not results:
            continue
        n_correct = sum(r.validity_correct for r in results)
        n_approved = sum(
            1 for r in results
            if r.parsed_response and r.parsed_response.validity == Validity.VALID
        )
        rows.append({
            "model": model_label,
            "strategy": strat,
            "trap_accuracy_pct": round(100 * n_correct / len(results), 1),
            "approved_as_valid": n_approved,  # the dangerous failure mode
        })

trap_df = pd.DataFrame(rows).sort_values(["approved_as_valid", "trap_accuracy_pct"],
                                          ascending=[True, False]).reset_index(drop=True)
trap_df

Trap cases: 16


,model,strategy,trap_accuracy_pct,approved_as_valid
0,Haiku 4.5,minimal,100.0,0
1,Haiku 4.5,zero_shot,100.0,0
2,Haiku 4.5,cot,100.0,0
3,Sonnet 4.5,minimal,100.0,0
4,Sonnet 4.5,default,100.0,0
5,Sonnet 4.5,zero_shot,100.0,0
6,Sonnet 4.5,few_shot,100.0,0
7,Haiku 4.5,default,93.8,0
8,Haiku 4.5,few_shot,93.8,0
9,GPT-4o,zero_shot,93.8,0


**Key observations from the trap table:**

- **Claude Sonnet 4.5 catches all 16 traps under every prompting strategy** —
  the strongest trap-detection performance of any model. This is the case for
  treating Sonnet 4.5 as the "maximum safety" recommendation.
- For **GPT-4o**, only `minimal` and `default` approve any traps; `zero_shot`,
  `few_shot`, and `chain_of_thought` all hit 0 approvals. (CoT and few-shot
  still mark some traps as "ambiguous" rather than "invalid" — those count
  against trap accuracy but are not dangerous approvals.)
- **Sonnet 4.6 is *worse* than 4.5 on trap detection** despite winning on
  overcaution — a real trade-off worth knowing if you're using Sonnet 4.6
  for high-stakes audits.

The four specific traps that the `default` GPT-4o prompt let through illustrate
*what kind of language fools the model*:

| Trap case | Bug | Reassuring phrase |
|---|---|---|
| `fvt_lag_from_datadate` | Fixed 6-month lag from fiscal end varies by firm; should use `rdq` / filing date | "conservative lag avoids look-ahead bias" |
| `fvt_dlret_replaces_ret` | Replaces RET with DLRET; should compound `(1+RET)*(1+DLRET) − 1` | "following best practices" |
| `fvt_adjusted_price_level` | CRSP split-adjusted prices used for a `< $5` level filter | "ensure historical comparability" |
| `fvt_restated_despite_lag` | 2026 Compustat download contains restated values not available historically | "lag fundamentals conservatively" |

In [15]:
# Interactive Plotly version (replaces the static PNG previously embedded here).
from generate_plotly_charts import (
    fig_leaderboard, fig_safety_overcaution, fig_trap_cases,
    fig_v6, fig_aa_omniscience,
    build_leaderboard_df, build_trap_df, build_v6_df, build_aa_df,
)

_df = build_trap_df(RESULTS_DIR)
_fig = fig_trap_cases(_df)
_fig

The middle panel above (false-invalid rate vs strategy) shows the clearest
pattern: **Sonnet 4.6 is essentially flat across all strategies**, while
Haiku 4.5 and Sonnet 4.5 only become well-calibrated when given more elaborate
prompts. GPT-4o sits between them — calibration is decent at minimal and gets
slightly better with few-shot.

## §11 V6: Isolating Temporal Reasoning from Domain Knowledge

V5 left one question unanswered: when a model fails to flag a buggy backtest,
is it failing at temporal reasoning, at finance-domain knowledge, or both?
V6 introduces two control batteries that decouple the two axes:

- **15 chronology cases** — pure temporal ordering, no real tickers or filings,
  just timestamps and the question "did event A happen before event B?"
- **12 code cases** — the same PIT violation taxonomy expressed as Python
  snippets instead of prose; tests whether models can audit code semantics.

The cell below loads results from both batteries and computes overall + valid +
invalid accuracy per model. The story it tells: overcaution generalizes
beyond finance, and the prompt-paradox inverts on code.

In [16]:
# V6 chronology breakdown: overall / valid / invalid accuracy per model.
chrono_cases = get_chronology_cases_only()
code_cases = get_code_cases_only()


def split_accuracies(results: list[ScoredResponse], cases) -> dict:
    case_map = {c.id: c for c in cases}
    parsed = [r for r in results if r.parse_success and r.case_id in case_map]
    valid_r = [r for r in parsed if case_map[r.case_id].expected_validity == Validity.VALID]
    invalid_r = [r for r in parsed if case_map[r.case_id].expected_validity == Validity.INVALID]
    return {
        "overall_pct":  round(100 * sum(r.validity_correct for r in parsed) / max(len(parsed), 1), 1),
        "valid_pct":    round(100 * sum(r.validity_correct for r in valid_r) / max(len(valid_r), 1), 1),
        "invalid_pct":  round(100 * sum(r.validity_correct for r in invalid_r) / max(len(invalid_r), 1), 1),
    }


v6_configs = [
    ("GPT-4o (Generic)",     "v6_chronology_gpt4o_generic.jsonl",     "code_cases_gpt4o_generic.jsonl"),
    ("GPT-4o (Specialized)", "v6_chronology_gpt4o_specialized.jsonl", "code_cases_gpt4o_specialized.jsonl"),
    ("Claude Sonnet 4.5",    "v6_chronology_claude_sonnet.jsonl",     "code_cases_claude_sonnet.jsonl"),
]
rows = []
for label, chrono_file, code_file in v6_configs:
    chrono_res = load_results_jsonl(RESULTS_DIR / chrono_file)
    code_res = load_results_jsonl(RESULTS_DIR / code_file)
    rows.append({
        "config":                label,
        "chrono_overall_pct":    split_accuracies(chrono_res, chrono_cases)["overall_pct"],
        "chrono_valid_pct":      split_accuracies(chrono_res, chrono_cases)["valid_pct"],
        "chrono_invalid_pct":    split_accuracies(chrono_res, chrono_cases)["invalid_pct"],
        "code_overall_pct":      split_accuracies(code_res, code_cases)["overall_pct"],
        "code_bug_detect_pct":   split_accuracies(code_res, code_cases)["invalid_pct"],
        "code_trap_valid_pct":   split_accuracies(code_res, code_cases)["valid_pct"],
    })
pd.DataFrame(rows)

,config,chrono_overall_pct,chrono_valid_pct,chrono_invalid_pct,code_overall_pct,code_bug_detect_pct,code_trap_valid_pct
0,GPT-4o (Generic),80.0,50.0,100.0,66.7,87.5,25.0
1,GPT-4o (Specialized),80.0,50.0,100.0,58.3,87.5,0.0
2,Claude Sonnet 4.5,86.7,66.7,100.0,66.7,100.0,0.0


**What V6 contributes.** On chronology, every model catches 100% of the
invalid (anachronistic) sequences but only 50-67% of the valid ones. On
code, Claude catches 100% of bugs but flags every correct code snippet as
broken (0% trap-valid). Two takeaways:

1. **Overcaution is not finance-specific.** Stripping away CRSP semantics and
   ticker history does not improve calibration on valid cases. The valid /
   invalid asymmetry observed across V1-V5 is a more general behavior.
2. **The prompt-paradox flips on code.** Where on prose the specialized
   prompt buys safety, on code Claude is the all-around best bug-catcher
   but also the most aggressive false-flagger.

In [17]:
# Interactive Plotly version (replaces the static PNG previously embedded here).
from generate_plotly_charts import (
    fig_leaderboard, fig_safety_overcaution, fig_trap_cases,
    fig_v6, fig_aa_omniscience,
    build_leaderboard_df, build_trap_df, build_v6_df, build_aa_df,
)

_df = build_v6_df(RESULTS_DIR)
_fig = fig_v6(_df)
_fig

## §12 AA-Omniscience: Does the BLD Prompt Transfer to Factual Recall?

The BLD `finance_auditor` system prompt encodes principles about *temporal
reasoning* in finance (identifier validity, information availability, universe
construction). Does it also help with factual recall on finance regulatory
questions? To find out, we ran the same prompt on **100 finance questions
from the AA-Omniscience benchmark** — questions like "Which paragraph of ASC
606 covers contract modifications?" — scored using the AA-Omniscience
**Omniscience Index** (OI):

$$\text{OI} = 100 \cdot \frac{c - i}{c + p + i + a}$$

where $c$, $p$, $i$, $a$ are counts of CORRECT, PARTIALLY_CORRECT, INCORRECT,
and NOT_ATTEMPTED responses. Crucially, abstaining is neutral — making
abstention strictly better than guessing wrong.

In [18]:
# Compute OI and hallucination rate for each AA-Omniscience config.
aa_configs = [
    ("GPT-4o (Generic)",         "aa_omniscience_gpt4o_generic.jsonl"),
    ("GPT-4o (Specialized BLD)", "aa_omniscience_gpt4o_specialized.jsonl"),
    ("Claude Sonnet (Specialized BLD)", "aa_omniscience_claude_sonnet.jsonl"),
]
rows = []
for label, fname in aa_configs:
    path = RESULTS_DIR / fname
    grades = []
    with jsonlines.open(path) as r:
        for rec in r:
            grades.append(rec["grade"])
    n = len(grades)
    c = sum(1 for g in grades if g == "CORRECT")
    p = sum(1 for g in grades if g == "PARTIALLY_CORRECT")
    i = sum(1 for g in grades if g == "INCORRECT")
    a = sum(1 for g in grades if g == "NOT_ATTEMPTED")
    halluc = i / (p + i + a) if (p + i + a) > 0 else 0.0
    rows.append({
        "config":            label,
        "n":                 n,
        "correct":           c,
        "partial":           p,
        "incorrect":         i,
        "abstained":         a,
        "oi_index":          round(aa_omniscience_index(grades), 1),
        "accuracy_pct":      round(100 * c / n, 1),
        "halluc_rate_pct":   round(100 * halluc, 1),
    })
aa_df = pd.DataFrame(rows)
aa_df

,config,n,correct,partial,incorrect,abstained,oi_index,accuracy_pct,halluc_rate_pct
0,GPT-4o (Generic),100,33,4,63,0,-30.0,33.0,94.0
1,GPT-4o (Specialized BLD),100,31,5,62,2,-31.0,31.0,89.9
2,Claude Sonnet (Specialized BLD),100,35,6,47,12,-12.0,35.0,72.3


In [19]:
# Interactive Plotly version (replaces the static PNG previously embedded here).
from generate_plotly_charts import (
    fig_leaderboard, fig_safety_overcaution, fig_trap_cases,
    fig_v6, fig_aa_omniscience,
    build_leaderboard_df, build_trap_df, build_v6_df, build_aa_df,
)

_df = build_aa_df(RESULTS_DIR)
_fig = fig_aa_omniscience(_df)
_fig

**The cross-benchmark finding.** GPT-4o leads on BLD (temporal reasoning)
but Claude leads on AA-Omniscience (factual recall + calibration). Domain
priming via the BLD prompt makes essentially **zero difference** on factual
recall — GPT-4o Generic (-30 OI) ≈ GPT-4o Specialized (-31 OI). Claude's
better OI is driven primarily by *willingness to abstain* (12 abstentions
vs 0-2 for GPT-4o), not by knowing more facts.

This confirms the project's framing: **temporal reasoning and factual recall
are orthogonal capabilities**. A model strong at one is not necessarily
strong at the other, and a prompt that improves one does not transfer to the
other. Full topic-level breakdown and methodology notes live in
[`writeup/AAOmniscience_results.md`](../writeup/AAOmniscience_results.md).

## §13 Practical Recommendations + Limitations

| Priority | Recommendation |
|---|---|
| **Best overall accuracy** | GPT-4o + chain-of-thought (85.1% on V5) |
| **Maximum safety (no false approvals)** | Claude Sonnet 4.5 (catches all 16 traps, every strategy) |
| **Least overcautious** | GPT-4o + few-shot (9.8% false-invalid) |
| **Most balanced calibration** | Claude Sonnet 4.6 (≤15% false-invalid across every strategy) |
| **Best repair suggestions** | Claude Sonnet 4.5 (~67% correct repairs on V4 sample) |
| **Complex methodology with reassuring language** | Specialized (`zero_shot`) or `few_shot` prompt |
| **Simple / obvious cases** | `minimal` or `default` prompt |

### Limitations

1. **Five model variants tested** — GPT-4o, GPT-4o-mini, Claude Sonnet 4.5,
   Sonnet 4.6, Haiku 4.5. Reasoning-tier models (o1, o3, Claude Opus) are out
   of scope.
2. **Five prompting strategies tested** — `minimal`, `default`, `zero_shot`,
   `few_shot`, `chain_of_thought`. More elaborate strategies (self-critique,
   multi-pass, tool use) could yield further gains.
3. **Ground truth requires domain expertise** — some `trap_valid` cases have
   debatable answers; see the labeling protocol in `writeup/index.md`.
4. **Benchmark size** — 141 main cases (V5) plus 27 V6 cases covers the major
   patterns but is not exhaustive.
5. **V6 batteries are small** — 15 chronology and 12 code cases are enough
   to expose the overcaution pattern but not to make fine-grained claims.
6. **No fine-tuning** — using base model capabilities only.
7. **English only** — all prompts and cases are in English.

## §14 Reproducibility

The repository is structured so that this notebook's tables and figures can
be rebuilt from scratch:

- **No-API rebuild:** `python scripts/reproduce.py` regenerates aggregate
  metrics and figures from committed JSONL files, runs the unit tests, and
  smoke-tests the AA-Omniscience loader. ~30 seconds, no API cost.
- **Core V4/V5/V6 live rerun:** `python scripts/reproduce.py --full-api`
  reruns the original 3-config × V4 + V5-trap + V6 batteries. ~20 minutes,
  ~$10 of API credit.
- **Extended 4-model × 5-strategy sweep:** the scripts under
  `scripts/reproduction/run_prompting_comparison.py`,
  `run_claude_prompting_comparison.py`, `run_gpt4o_base_prompts.py`,
  `run_gpt4o_v6only.py`, `run_aa_omniscience.py`, and
  `run_baseline_evaluation.py` must be run manually. Total: ~$25-30 of
  API credit, ~75 minutes wall-clock.

The canonical aggregate summary is committed at
[`outputs/results/ALL_EVALUATION_RESULTS.txt`](../outputs/results/ALL_EVALUATION_RESULTS.txt).
The full writeup is at [`writeup/index.md`](../writeup/index.md).

---

*End of report.*